# SUPP Survey Analysis with Claude Skills

Use Claude's Skills feature to analyze **Student Substance Use and Peer Perception (SUPP) Survey** data from Prevention Education Solutions.

This notebook demonstrates:
1. Uploading and using a **custom skill** for SUPP survey domain expertise
2. Combining the custom skill with **Anthropic skills** (xlsx, pptx, pdf) for report generation
3. Processing Typeform CSV exports and comparing against **MTF national benchmarks**
4. Generating professional deliverables for school administrators

## Table of Contents

1. [Setup](#setup)
2. [Upload the SUPP Skill](#upload-skill)
3. [Analyze Survey Data → Excel Dashboard](#excel-dashboard)
4. [Generate Executive Summary → PowerPoint](#pptx-summary)
5. [Create Full Report → PDF](#pdf-report)
6. [Cleanup](#cleanup)

## 1. Setup {#setup}

### Prerequisites
- Completed setup from [Notebook 1](01_skills_introduction.ipynb)
- `.env` file with your `ANTHROPIC_API_KEY`
- Survey data CSV (Typeform export) and MTF benchmark CSV

### File Locations

Update these paths to match your data:
- **Survey CSV:** Typeform export of SUPP responses
- **MTF CSV:** Monitoring the Future national benchmark data
- **Custom Skill:** `../custom_skills/supp-survey-analysis/`

In [ ]:
import os
import sys
from pathlib import Path

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent))

from anthropic import Anthropic
from dotenv import load_dotenv

from file_utils import download_all_files, extract_file_ids, print_download_summary

# Load environment variables
load_dotenv(Path.cwd().parent / ".env")

API_KEY = os.getenv("ANTHROPIC_API_KEY")
MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")

if not API_KEY:
    raise ValueError("ANTHROPIC_API_KEY not found. See Notebook 1 for setup instructions.")

# Initialize clients
client = Anthropic(api_key=API_KEY)
client_skills = Anthropic(
    api_key=API_KEY, default_headers={"anthropic-beta": "skills-2025-10-02"}
)

# Output directory
OUTPUT_DIR = Path.cwd().parent / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Custom skill path
SKILL_PATH = str(Path.cwd().parent / "custom_skills" / "supp-survey-analysis")

print(f"\u2713 API key loaded")
print(f"\u2713 Model: {MODEL}")
print(f"\u2713 Skill path: {SKILL_PATH}")
print(f"\u2713 Output dir: {OUTPUT_DIR}")

### Configure Your Survey

Update these variables for your specific school and survey data:

In [ ]:
# =============================================================================
# CONFIGURE THESE FOR YOUR SURVEY
# =============================================================================

SCHOOL_NAME = "International School of Bangkok"  # Your school name
SURVEY_DATE = "January 2026"                     # When the survey was administered
GRADES = ["9th", "10th", "11th", "12th"]         # Grade levels included

# Path to your Typeform CSV export
SURVEY_CSV = os.path.expanduser(
    "~/Desktop/2026 SUPP files/IST SUPP Data.csv"
)

# Path to MTF benchmark data
MTF_CSV = os.path.expanduser(
    "~/Desktop/2026 SUPP files/MTF2024.csv"
)

# Verify files exist
for label, path in [("Survey CSV", SURVEY_CSV), ("MTF CSV", MTF_CSV)]:
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"\u2713 {label}: {path} ({size_kb:.1f} KB)")
    else:
        print(f"\u2717 {label}: NOT FOUND at {path}")
        print(f"   Update the path above to point to your data file.")

## 2. Upload the SUPP Skill {#upload-skill}

First, let's upload the custom SUPP survey analysis skill to the Skills API.
This packages the domain expertise (survey sections, MTF comparisons, K10 scoring, risk categorization) so Claude can use it during analysis.

In [ ]:
from anthropic.lib import files_from_dir

# Upload the custom skill
supp_skill = client_skills.beta.skills.create(
    display_title="SUPP Survey Analysis",
    files=files_from_dir(SKILL_PATH),
)

SUPP_SKILL_ID = supp_skill.id

print(f"\u2713 Skill uploaded successfully!")
print(f"   ID: {SUPP_SKILL_ID}")
print(f"   Title: {supp_skill.display_title}")
print(f"   Version: {supp_skill.latest_version}")
print(f"   Source: {supp_skill.source}")

In [ ]:
# Verify the skill contents
version_info = client_skills.beta.skills.versions.retrieve(
    skill_id=SUPP_SKILL_ID, version=supp_skill.latest_version
)

print(f"Skill Name: {version_info.name}")
print(f"Description: {version_info.description[:200]}...")
print(f"\nFiles in skill:")
for entry in version_info.directory:
    print(f"   {entry.path} ({entry.type})")

### Upload Survey Data Files

Upload the survey CSV and MTF benchmark data so Claude can access them during code execution.

In [ ]:
# Upload survey data file
survey_file = client.beta.files.upload(
    file=("survey_data.csv", open(SURVEY_CSV, "rb"), "text/csv"),
)
print(f"\u2713 Survey data uploaded: {survey_file.id} ({survey_file.size_bytes / 1024:.1f} KB)")

# Upload MTF benchmark data
mtf_file = client.beta.files.upload(
    file=("mtf_benchmarks.csv", open(MTF_CSV, "rb"), "text/csv"),
)
print(f"\u2713 MTF data uploaded: {mtf_file.id} ({mtf_file.size_bytes / 1024:.1f} KB)")

## 3. Analyze Survey Data and Create Excel Dashboard {#excel-dashboard}

Combine the custom SUPP skill with the Anthropic `xlsx` skill to:
- Process the raw survey CSV
- Calculate prevalence rates and demographics
- Compare against MTF benchmarks
- Generate a formatted Excel workbook with multiple tabs

**Expected time: 2-3 minutes**

In [ ]:
excel_response = client.beta.messages.create(
    model=MODEL,
    max_tokens=8192,
    container={
        "skills": [
            {"type": "custom", "skill_id": SUPP_SKILL_ID, "version": "latest"},
            {"type": "anthropic", "skill_id": "xlsx", "version": "latest"},
        ]
    },
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": f"""Analyze this SUPP survey data for {SCHOOL_NAME} ({SURVEY_DATE}).
Grades surveyed: {', '.join(GRADES)}.

The first file is the Typeform CSV export of student survey responses.
The second file is the MTF (Monitoring the Future) 2024 national benchmark data.

Create an Excel workbook with these tabs:

1. **Demographics** - Response counts and percentages for sex, grade, age, academic performance
2. **Substance Prevalence** - Lifetime use rates for all substances, compared to MTF national norms
3. **Alcohol Detail** - Frequency, quantity, past-12-month, past-30-day breakdowns by grade
4. **Risk Categories** - Non-drinker/low-risk/high-risk distribution by grade
5. **School Environment** - Agree/disagree percentages for all culture statements
6. **K10 Mental Health** - Distress level distribution, mean scores by grade and risk category
7. **Key Findings** - Top 10 most notable findings with MTF comparison bullets

Use professional formatting:
- Dark blue headers with white bold text
- Alternating row shading
- Percentages to 1 decimal place
- Include charts on the Substance Prevalence and K10 tabs
"""},
                {"type": "container_upload", "file_id": survey_file.id},
                {"type": "container_upload", "file_id": mtf_file.id},
            ],
        }
    ],
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"],
)

print("Excel Analysis Response:")
print("=" * 80)
for content in excel_response.content:
    if content.type == "text":
        print(content.text)

print(f"\n\nToken Usage: {excel_response.usage.input_tokens} in, {excel_response.usage.output_tokens} out")

In [ ]:
# Download the Excel file
file_ids = extract_file_ids(excel_response)

if file_ids:
    results = download_all_files(
        client, excel_response, output_dir=str(OUTPUT_DIR), prefix=f"supp_{SCHOOL_NAME.replace(' ', '_')}_"
    )
    print_download_summary(results)
else:
    print("No files found. Check the response above for errors.")

## 4. Generate Executive Summary Presentation {#pptx-summary}

Create a PowerPoint deck for presenting findings to school administrators.

**Expected time: 1-2 minutes**

In [ ]:
pptx_response = client.beta.messages.create(
    model=MODEL,
    max_tokens=8192,
    container={
        "skills": [
            {"type": "custom", "skill_id": SUPP_SKILL_ID, "version": "latest"},
            {"type": "anthropic", "skill_id": "pptx", "version": "latest"},
        ]
    },
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": f"""Create an executive summary PowerPoint presentation of the SUPP survey results
for {SCHOOL_NAME} ({SURVEY_DATE}). Grades: {', '.join(GRADES)}.

The first file is the survey data CSV, the second is MTF benchmarks.

Create a 5-slide presentation:

Slide 1: Title slide
- "Student Substance Use & Peer Perception Survey"
- Subtitle: "{SCHOOL_NAME} - {SURVEY_DATE}"
- "Prepared by Prevention Education Solutions"

Slide 2: Survey Overview
- Total respondents, grade breakdown
- Response rate and data quality notes

Slide 3: Key Substance Use Findings
- Top 5 substances by prevalence
- Bar chart comparing school rates vs MTF national norms
- Above/below norms callouts

Slide 4: School Culture & Mental Health
- Key school environment findings (pressure, respect for abstainers)
- K10 distress level distribution

Slide 5: Recommendations
- 3-4 data-driven recommendations based on the findings
- Areas where the school is above national norms (concern)
- Areas where the school is below national norms (strength)

Use professional, clean formatting. Navy blue and white color scheme.
"""},
                {"type": "container_upload", "file_id": survey_file.id},
                {"type": "container_upload", "file_id": mtf_file.id},
            ],
        }
    ],
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"],
)

print("PowerPoint Response:")
print("=" * 80)
for content in pptx_response.content:
    if content.type == "text":
        print(content.text)

print(f"\nToken Usage: {pptx_response.usage.input_tokens} in, {pptx_response.usage.output_tokens} out")

In [ ]:
# Download the PowerPoint file
file_ids = extract_file_ids(pptx_response)

if file_ids:
    results = download_all_files(
        client, pptx_response, output_dir=str(OUTPUT_DIR), prefix=f"supp_{SCHOOL_NAME.replace(' ', '_')}_"
    )
    print_download_summary(results)
else:
    print("No files found. Check the response above for errors.")

## 5. Generate PDF Summary Report {#pdf-report}

Create a formatted PDF report with key findings, tables, and charts.

**Expected time: 1-2 minutes**

In [ ]:
pdf_response = client.beta.messages.create(
    model=MODEL,
    max_tokens=8192,
    container={
        "skills": [
            {"type": "custom", "skill_id": SUPP_SKILL_ID, "version": "latest"},
            {"type": "anthropic", "skill_id": "pdf", "version": "latest"},
        ]
    },
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": f"""Create a PDF summary report of the SUPP survey results
for {SCHOOL_NAME} ({SURVEY_DATE}). Grades: {', '.join(GRADES)}.

The first file is the survey data CSV, the second is MTF benchmarks.

Structure the report as:

1. **Header** - "SUPP Survey Summary Report" with school name, date, prepared by Prevention Education Solutions

2. **Survey Overview** - Total N, grade distribution, data quality (contradictions removed)

3. **Substance Use Summary Table** - All substances with lifetime prevalence, compared to MTF norms.
   Use above/at/below indicators.

4. **Alcohol Deep Dive** - Frequency, quantity, past-30-day rates by grade.
   Include top 5 most reported consequences.

5. **Perception vs Reality** - What students think peers use vs actual reported rates.
   Highlight the perception gap.

6. **School Environment Highlights** - Top 3 positive and top 3 concerning findings.

7. **Mental Health (K10)** - Distress level distribution. Correlation with substance use risk category.

8. **Key Recommendations** - 5 data-driven action items.

Formatting:
- Georgia font for headings, clean sans-serif for body
- Dark blue (#3333cc) section headers
- Professional tables with alternating row shading
- Include the school name and date in the header/footer
"""},
                {"type": "container_upload", "file_id": survey_file.id},
                {"type": "container_upload", "file_id": mtf_file.id},
            ],
        }
    ],
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"],
)

print("PDF Report Response:")
print("=" * 80)
for content in pdf_response.content:
    if content.type == "text":
        print(content.text)

print(f"\nToken Usage: {pdf_response.usage.input_tokens} in, {pdf_response.usage.output_tokens} out")

In [ ]:
# Download the PDF file
file_ids = extract_file_ids(pdf_response)

if file_ids:
    results = download_all_files(
        client, pdf_response, output_dir=str(OUTPUT_DIR), prefix=f"supp_{SCHOOL_NAME.replace(' ', '_')}_"
    )
    print_download_summary(results)

    # Verify PDF integrity
    for result in results:
        if result["success"]:
            with open(result["output_path"], "rb") as f:
                header = f.read(5)
                if header == b"%PDF-":
                    print(f"\nPDF is valid: {result['output_path']}")
                else:
                    print(f"\nFile may not be a valid PDF: {result['output_path']}")
else:
    print("No files found. Check the response above for errors.")

## 6. Cleanup {#cleanup}

Delete the uploaded skill and files when you're done experimenting.

**Only run this when you no longer need the skill.**

In [ ]:
# Uncomment to delete the custom skill
# (Uploaded files are cleaned up automatically)

# from skill_utils import delete_skill
# delete_skill(client_skills, SUPP_SKILL_ID)

print("To delete the skill, uncomment the lines above and re-run this cell.")
print(f"\nSkill ID to delete: {SUPP_SKILL_ID}")

## Summary

In this notebook you:

1. **Created a custom SUPP skill** — Packaged survey domain expertise (sections, MTF comparisons, K10 scoring, risk categories) as a reusable skill
2. **Generated an Excel dashboard** — Multi-tab workbook with demographics, prevalence rates, risk categories, and MTF comparisons
3. **Built a PowerPoint deck** — Executive summary presentation for school administrators
4. **Produced a PDF report** — Formatted summary with tables, findings, and recommendations

### Using This for New Schools

To run this for a different school:
1. Update `SCHOOL_NAME`, `SURVEY_DATE`, and `GRADES` in the config cell
2. Update `SURVEY_CSV` to point to the new Typeform export
3. Re-run all cells

The custom skill only needs to be uploaded once — you can reuse the same `SUPP_SKILL_ID` across multiple schools.

### Next Steps

- Modify the SUPP skill in `custom_skills/supp-survey-analysis/` to add new analysis capabilities
- Use `create_skill_version()` from `skill_utils.py` to update the uploaded skill
- Combine with the `docx` Anthropic skill to generate Word documents matching your existing SUPP template

### Resources

- [Skills Documentation](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/overview)
- [Custom Skills Best Practices](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/best-practices)
- [Notebook 3: Custom Skills Development](03_skills_custom_development.ipynb)